# v6 overlap40：FEC 单模块与 Gated+FEC 并行实验

账号1运行 DeepLab+FEC（C），账号2运行 Gated+FEC（A+C）。两者均使用 seed42、physical batch4、相同数据与训练超参，按 Val mIoU_fg 选模，不访问 Test。

## 1. 选择实验


In [ ]:
# 账号1：模块C单独，DeepLab + FEC。
CONFIG_FILE = 'v6_overlap40_deeplab_fec_batch4_seed42.json'
# 账号2：注释上一行，启用 A+C。
# CONFIG_FILE = 'v6_overlap40_gated_fec_batch4_seed42.json'

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'


## 2. 环境与代码


In [ ]:
import hashlib, importlib.metadata, importlib.util, json, shutil, subprocess, sys
from pathlib import Path
REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')
required = [('rasterio', 'rasterio'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import torch
assert torch.cuda.is_available(), '请在 Notebook settings 中开启 GPU'
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
config = json.loads((PROJECT_DIR / 'configs' / CONFIG_FILE).read_text(encoding='utf-8'))
assert config['module'] in {'deeplab_fec', 'gated_fec'}
assert config['batch_size'] == 4 and config['accum_steps'] == 1
assert config['boundary_weight'] == 0.0 and config['foreground_weight'] == 0.1
assert config['selection_metric'] == 'val_mIoU_fg'
assert config['automatic_test_evaluation'] is False
print('GPU:', torch.cuda.get_device_name(0))
print('Commit:', commit)
print(json.dumps(config, ensure_ascii=False, indent=2))


## 3. 核验 overlap40 数据


In [ ]:
import numpy as np
existing = [Path(path) for path in config['data_candidates'] if Path(path).is_dir()]
assert existing, '没有找到 overlap40 数据集，请 Add Input: yuanssy/datav6-overlap40'
DATA_ROOT = existing[0]
for split, expected in config['expected_tiles'].items():
    images = sorted([*(DATA_ROOT / split / 'image').glob('*.tif'), *(DATA_ROOT / split / 'image').glob('*.tiff')])
    masks = sorted([*(DATA_ROOT / split / 'mask').glob('*.tif'), *(DATA_ROOT / split / 'mask').glob('*.tiff')])
    assert len(images) == len(masks) == expected, (split, len(images), len(masks))
    assert {p.stem for p in images} == {p.stem for p in masks}
for name, expected_hash in config['expected_metadata_sha256'].items():
    actual_hash = hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, (name, actual_hash, expected_hash)
stats = json.loads((DATA_ROOT / 'normalization_stats.json').read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], config['expected_mean'], rtol=0, atol=1e-12)
assert np.allclose(stats['std'], config['expected_std'], rtol=0, atol=1e-12)
print('Data:', DATA_ROOT, '核验通过')


## 4. batch4 前向与反向冒烟


In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from models.module_models import build_module_model
from train_module_experiment import ExperimentLoss, training_outputs
model = build_module_model(config['module'], encoder_weights=None).cuda().train()
criterion = ExperimentLoss(config['module'], config['boundary_weight'], config['foreground_weight']).cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'])
scaler = torch.amp.GradScaler('cuda')
x = torch.randn(4, 5, 512, 512, device='cuda')
labels = torch.randint(0, 5, (4, 512, 512), device='cuda')
with torch.amp.autocast('cuda'):
    logits, boundary_logits, foreground_logits = training_outputs(model, x, config['module'])
    loss, parts = criterion(logits, labels, boundary_logits, foreground_logits)
assert torch.isfinite(loss) and foreground_logits is not None
scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
assert logits.shape == (4, 5, 512, 512)
print('Parameters:', f'{sum(p.numel() for p in model.parameters()):,}')
print('Smoke loss:', float(loss), parts)
del model, criterion, optimizer, scaler, x, labels, logits, boundary_logits, foreground_logits, loss
torch.cuda.empty_cache()


## 5. 正式训练


In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
assert not result_dir.exists(), f'结果目录已存在: {result_dir}'
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'train_module_experiment.py'),
    '--module', config['module'], '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT), '--run-name', config['run_name'],
    '--seed', str(config['seed']), '--epochs', str(config['epochs']),
    '--batch-size', str(config['batch_size']), '--accum-steps', str(config['accum_steps']),
    '--num-workers', str(config['num_workers']), '--learning-rate', str(config['learning_rate']),
    '--boundary-weight', str(config['boundary_weight']),
    '--foreground-weight', str(config['foreground_weight']),
    '--encoder-weights', config['encoder_weights'],
]
print(' '.join(command))
subprocess.check_call(command, cwd=PROJECT_DIR)


## 6. 查看并打包结果


In [ ]:
result = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
assert result['selection_metric'] == 'val_mIoU_fg' and result['test_evaluated'] is False
print(json.dumps(result, ensure_ascii=False, indent=2))
archive_path = shutil.make_archive(str(OUTPUT_ROOT / result_dir.name), 'zip', root_dir=result_dir)
print('请下载:', archive_path)
